TRACKING DI OGGETTI IN VIDEO: DALLA DETEZIONE ALLA PERSISTENZA TEMPORALE.

Smettiamo di vedere il mondo come una serie di foto statiche ed iniziamo a vederlo come un flusso continuo.
Finora abbiamo visto: cosa c'è in questa immagine.
Ora la sfida cambia: immmagina di guardare una partita di calcio, non basta sapere che c'è un pallone ma chi lo sta portando e dove sta andando.
Passiamo dalla visione istantanea alla memoria visiva.
Dobbiamo quindi dare memoria al computer.

Finora abbiamo visto come identificare oggetti in singole immagini statiche. Tuttavia, in un video, un oggetto non è solo una collezione di pixel in una posizione, ma un'entità che evolve nel tempo. La detenzione pura 'dimentica' tutto ciò che è successo nel frame precedente.
Il tracking, invece, aggiunge la dimensione di persistenza: permette di assegnare un ID univoco a un oggetto e seguirlo attraverso i fotogrammi, permetendo analisi come il conteggio di persone che entrano in una stanza o la stima della velocità di un veicolo.

YOLO, se usato solo come detector, guarda ogni frame di un video, come se fosse un'immagine indipendente. 

La domanda passa da:
detection: "Cosa vedo e dove si trova?"
a 
tracking: "Quello che vedo adesso, è lo stesso oggetto che vedevo nel frame precedente?"

questa è PERSISTENZA TEMPORALE

1.  Detection senza tracking
    Immgina un video di un'auto, yolo vedo tanti frame indipendenti tra di loro, tratta ogni immagine in modo a se stante. La sola detection non riesce a capire se è sempre la stessa automobile
2.  Detector + Tracker 
    Aggiungendo il tracker, ad ogni oggetto, viene associato un identificativo numerico persistente su tutti il video. L'oggetto si muove nel video, ed è questa la differenza tra normale object deteciont e multi-object-tracking
    alla macchina dell'esempio precednete viene associato un ID che si mantiene per tutto il video, è quindi possibile dire se la macchina del frame-1 è la stessa macchina del frame-2 o del frame-n. E' inoltre possibile capire la traiettoria, durante tutto il video, della stessa macchina

video -> frame -> object detector (es. yolo) - bouding boxes, classe, confidence -> TRACKER -> ID persistente -> traiettoria

YOLO trova gli oggetti il TRACKET collega nel tempo la detection

Questo si chiama DATA ASSOCIATION, cioè associazione tra detection appartenenti allo stesso oggetto nel tempo. SORT, per esempio, combina previsione del moviemento e algoritmo di asssegnazione per effetturare questa associazione in modo molto efficiente.

frame-1 auto id=1 x=100
frame-2 auto id=1 x=110
freme-3 ? 
il tracket può stimare:
    probabilità che l'auto id=1 si trovi a x=120 nel frame-3

Uno strumento classico per fare ciò, per fare previsione è Kalman Filter
Il concetto è:
posizione precedente + velocità stimata + nuova osservazione -> stima posizione attuale
SORT utilizza un filtro di Kalman per predire lo stato degli oggetti tra frame successivi.

Il tracking ci regala l'identità

Identità e Coerenza Temporale
I limiti computazionali della detenzione continua
Assegnare un ID univoco, significa poter contare quante persone entrano in un negozio, senza contare 10 volte la stessa persona che cammina avanti ed indietro.
C'è anche un vantaggio di efficienza computazionale, il tracking ci permette di utilizzare la storia passata dell'oggetto

La Pipeline di Tracking:
-   Input della Detenzione: il tracker riceve in input un set di bouding box generate da un detector come YOLO o SSD. Senza detecion iniziale, il tracker non ha nulla da seguire.
-   Associazione dei Dati: il cuore del tracking è capire quale box del frame attuale corrisponde a quale box del frame precedente, basandosi su prossimità o aspetto visivo
-   Predizione dello Stato: molti tracker avanzati tentano di 'indovinare' dove si troverà      
    l'oggetto nel frame successivo prima ancora di vederlo effettivamente.

Ma su quale principio si base questa scommessa sul futuro?

Analidi del Moto
Dinamica degli oggetti nel tempo
Mentre la detecion si concentra sulle caratteristiche semantiche dell'oggetto, il tracking si concentra sulla sua cinematica.
Nel mondo reale gli oggetti non si teletrasportano, hanno una inerzia una velocità e una direzione.
Il tracker usa questi dati, se sappiamo dove si trova un oggetto, e quanto velocemente si muoveva nei frame precedenti, la sua posizione futura non è un mistero ma un calcolo probabile.
Il tracker è l'arte di unire i puntini mantenendo la fluidità del movimento.
La continuità del moto è l'ipotesi fondamentale su cui si basano gli algoritmi di inseguimento
Considerando la posizione p al tempo t, il tracker assume che al tempo t+1 la posizione sarà in un intorno immediato della precedente.

Uno degli algoritmi che fa questa previsione è il Centroid Tracking

*** L'Algoritmo Centroid Tracking ***

Semplicità ed efficienza nell'associazione
Il Centroid Tracking è uno degli algoritmi di inseguimento più intuitivi ed efficaci per contesti in cui gli oggetti non si muovono in modo erratico. Si base esclusivamente sulla distanza geometrica tra i centri delle bounding box.
Il Centroid Tracking è il metodo più semplice per capire concretamente, come si passa dalla detection al tracking
L'idea è semplice: se un oggetto nel frame successivo si trova vicino a dove si trovava prima (frame precedente), probabilmente è lo stesso oggetto.
Calcolare il centro di massa di ogni oggetto rilevato e associarlo a quello pù vicino nel frame precedente che non sia ancora stato 
Ricuciamo ogni oggetto ad un singolo punto, il suo baricentro, è come guardare la folla dall'alto e vedere solo dei puntini luminosi, se un puntino si muove di poco è estremamente probabile che sia la stessa persona.
Non guarda il volto, il colore o le caratteristiche visive. Guarda principalmente la posizione del centro della bouding box

Come si muove questo algoritmo tra i fotogrammi?
- Calcolo dei Centroidi: estrazione del punto centrale (x,y) per ogni bounding box (per ogni      oggetto) prodotta dal detector
-  Matrice delle Distanze: calcolo della distanza euclidea tra tutti i nuovi centroidi e quelli degli oggetti già noti
- Associazione: selezione delle coppie con la distanza minima per mantenere la coerenza dell'identità. occoppiamo i centri più vicini tra loro
- Registrazione:  per ilcentro che non ha compagni, creazioe di un nuovo ID per ogni centroide che non trova corrispondenze valide nei dati esistenti.

Ma cosa succede se un oggetto corre troppo o se il video scatta?

Distanza Euclidea e Soglie
Metrica di prossimità spaziale
La distanza euclidea è il criterio principale per l'associazione. Tuttavia, se un oggetto si muove troppo velocemente o il frame rate è troppo basso, l'associazione potrebbe fallire o scambiare ID tra oggetti vicini.
Viene solitamente introdotta una soglia massima di distanza oltra la quele l'associazione non è ritenuta valida, portando alla creazione di un nuovo oggetto.
Senza questa soglia se un oggetto uscisse a sinistra ed un altro oggetto entrerebbe a destra contemporaneamente, il tracker potrebbe scambiarli.

Finotra abbiamo ragionato in un contesto in cui tutto sia sempre visibile, ma sappiamo che il mondo è pieno di ostacoli

Gestione della Scomparsa Temporanea
Robustezza contro occlusioni e rumore.
Nel mondo reale, gli oggetti non appaiono e scompaiono magicamente, ma possono essere temporaneamente coperti da altri oggetti (occlusione) o il detector potrebbe fallire nel rilevarli per qualche fotogramma a causa di riflessi o sfocatura.
Se una persona cammina dietro ad un pilastro, il tracket non vedendolo azzera il suo id, quando la persona esce dal pilastro il tracker assegnerebbe un nuovo ID, perdendo la continuità.
Un buon sistema di tracking deve essere 'resiliente' a paziente: deve concedere una seconda possibilità agli oggetti che spariscono aspettando un tempo ragionevaole prima di dichiararli ufficialmente persi.

Ma come implementiamo questa pazienza nel nostro codice Python?
Usiamo il concetto di Max Disappeared

Strategia di Resilienza (paziente)
Mantenere l'identità nel tempo
- Max Disappeared: parametro che definisce quanti frame consecutivi un oggetto può mancare prima di essere rimosso dal sistema. Numero di chance che diamo ad un oggetto prima di rimuoverlo dal database. Durante questo periodo di attesa l'oggetto è in un limbo, non lo vediamo ma conseviamo il suo ID sperando in una nuovo associazione.
- Deregistrazione: la procedurea formale di eliminazione di un ID dal database attivo quando la soglia di sparizione è superata
- Occlusione Parziale: situazione in cui l'oggetto è ancora visibile ma la bounding box cambia drasticamente forma o posizione.
- Re-Identification: capacità di riconoscere un oggetto che riappare dopo un lungo periodo, sabbene sia una sfida avanzata per il centroid tracking

Come funziona il contatore interno che deve gestire questa attesa?

Il Contatore di Sparizione
- Incremento del contatore: per ogni oggetto che non trova una corrispondenza nel frame corrente, il suo contatore 'disappeared' viene incrementato di uno
- Reset del Contatore: se l'oggetto viene ritrovato nel frame successivo (o entro il limite impostato), il sou contatore viene immediatamente azzerato, mantenendo lo stesso ID
- Trade-off dalla Soglia: una soglia troppo alta mantiene ID morti troppo a lungo (falsi positivi); una troppo bassa causa frequenti cambi di ID (id switching) per oggetti reali.

Tutto questo ha dei limiti

Verso Tracker più Avanzati
Limiti del centroide e soluzioni SOTA
Mentre il centroid tracking è veloce, fallisce se i box di sovrappongono. Tracket moderni come SORT integrano il Filtro Kalman per predire il moto futuro e feature vettoriali  (embeddings) per riconoscere l'aspetto visivo.
SORT calcola la traiettoria più probabile basandosi sulla fisica
SORT quindi aggiunge  la previsione indicando la posizione prevista.
Centroid Tracket -> posizione attuale
SORT -> posizione + movimento previsto
Il Filtro di Kalman permette di mantenere una traccia fluida anche quando l'osservazione è rumorosa, modellando l'incertezza del sensore e del modello di moto.

Deep SORT
Deep SORT agginge anche l'aspetto visivo.
Quindi:
Centroid: dove sei?
SORT: dove sei + dove stai andando
Deep SORT: dove sei + dove stai andando + come sei fatto

In [ ]:
import os

"""
Pipeline di Video Tracking con YOLOv8/YOLOv11.

Questo script implementa un flusso di lavoro completo per:
1. Scaricare un video da un URL remoto in un file temporaneo.
2. Inizializzare un modello YOLO con backend ottimizzato.
3. Eseguire il tracking degli oggetti (Object Tracking) con identificativi persistenti.
4. Salvare il risultato elaborato in un nuovo file video.

Dipendenze: ultralytics, opencv-python, requests
"""

# --- CONFIGURAZIONE AMBIENTE ---
# Impostiamo il backend di Keras a "torch" (PyTorch). 
# YOLO di Ultralytics usa internamente PyTorch; forzare l'ambiente aiuta la coerenza
# delle prestazioni e l'allocazione della memoria GPU/CPU in ambienti misti.
os.environ["KERAS_BACKEND"] = "torch"

import cv2
import requests
import tempfile
from ultralytics import YOLO

class YOLOVideoTracker:
    """
    Gestore del tracking video che incapsula la logica di YOLO.
    
    Utilizza il sistema di tracking nativo di Ultralytics, che integra
    algoritmi come ByteTrack e BoT-SORT per mantenere l'identità degli
    oggetti (ID) attraverso i frame.
    """
    def __init__(self, model_variant='yolov8n.pt'):
        """
        Inizializza il modello YOLO.
        :param model_variant: Nome del file del modello (es. 'yolov8n.pt' per nano, 'yolov8s.pt' per small).
        """
        print(f"Inizializzazione YOLO con Tracking Nativo: {model_variant}...")
        # Carica il modello pre-addestrato. Se model_preset esiste nel contesto locale (es. da script esterni), lo usa.
        self.model = YOLO(model_variant)

    def download_video(self, url):
        """
        Scarica un video da un URL e lo salva in un file temporaneo.
        Metodo necessario perché OpenCV (cv2.VideoCapture) legge meglio da file locali che da stream HTTP diretti.
        
        :param url: Link diretto al file video.
        :return: Path del file temporaneo creato.
        """
        print(f"Scaricamento video: {url}")
        headers = {'User-Agent': 'Mozilla/5.0'} # Header per evitare blocchi da parte del server
        response = requests.get(url, headers=headers, stream=True)
        response.raise_for_status() # Genera un errore se il download fallisce
        
        # Creiamo un file temporaneo che verrà rimosso alla fine dell'elaborazione
        temp_video = tempfile.NamedTemporaryFile(delete=False, suffix='.mp4')
        for chunk in response.iter_content(chunk_size=8192):
            temp_video.write(chunk)
        temp_video.close()
        return temp_video.name

    def process_video(self, video_url, output_path="output_simple_track.mp4"):
        """
        Esegue l'intera pipeline di processing sul video.
        
        :param video_url: URL di origine.
        :param output_path: Nome del file video prodotto.
        """
        video_path = self.download_video(video_url)
        cap = cv2.VideoCapture(video_path)
        
        # Estrazione metadati necessari per configurare il file di output (stesse dimensioni e velocità del sorgente)
        width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
        height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
        fps = int(cap.get(cv2.CAP_PROP_FPS))
        
        # Configurazione del VideoWriter: 'mp4v' è il codec standard per i file .mp4
        fourcc = cv2.VideoWriter_fourcc(*'mp4v')
        out = cv2.VideoWriter(output_path, fourcc, fps, (width, height))

        print("Elaborazione video con Tracking nativo (ByteTrack)...")
        frame_count = 0

        while cap.isOpened():
            # Legge il frame successivo
            ret, frame = cap.read()
            # Se il video finisce o superiamo i 100 frame (per test rapido), ci fermiamo
            if not ret or frame_count > 100: 
                break

            # --- LOGICA DI TRACKING ---
            # .track() è il metodo di Ultralytics che combina rilevamento e associazione temporale.
            # Parametri chiave:
            # - persist=True: Fondamentale. Comunica al modello che il frame fa parte di una sequenza
            #   e deve mantenere gli ID degli oggetti visti in precedenza.
            # - conf=0.3: Soglia di confidenza. Ignora rilevamenti con probabilità inferiore al 30%.
            # - iou=0.5: Intersection Over Union. Gestisce la soppressione dei box sovrapposti.
            # - tracker="bytetrack.yaml": Specifica l'algoritmo di tracking (ByteTrack è ottimo per fluidità).
            results = self.model.track(
                source=frame, 
                persist=True, 
                conf=0.3, 
                iou=0.5, 
                tracker="bytetrack.yaml",
                verbose=False
            )

            # Il metodo .plot() disegna automaticamente i box, le etichette delle classi (es. "person")
            # e l'ID univoco dell'oggetto (es. "1", "2") sul frame.
            annotated_frame = results[0].plot()

            # Aggiunge il frame annotato al video finale
            out.write(annotated_frame)
            
            frame_count += 1
            if frame_count % 20 == 0:
                print(f"Processati {frame_count} frame...")

        # Rilascio delle risorse hardware e chiusura dei file
        cap.release()
        out.release()
        
        # Pulizia: rimuoviamo il video temporaneo scaricato all'inizio
        if os.path.exists(video_path):
            os.unlink(video_path) 
            
        print(f"Processo concluso! Video salvato in: {output_path}")

# --- PUNTO DI INGRESSO (ENTRY POINT) ---
if __name__ == "__main__":
    # Inizializziamo l'elaboratore (usa il modello nano 'yolov8n.pt' per default)
    video_processor = YOLOVideoTracker()
    
    # URL di un video pubblico contenente persone, biciclette e auto per testare il tracking multi-classe
    VIDEO_URL = "https://raw.githubusercontent.com/intel-iot-devkit/sample-videos/master/person-bicycle-car-detection.mp4"
    
    # Avvio del processo
    video_processor.process_video(VIDEO_URL)